# Whole-Body Physics Signatures for Humanoid Gait Failure Ranking

## tl;dr

This notebook connects the implemented method to the frozen article result:

**robot-normalized gait sample → whole-body trajectory → phase physics
signature → independent closed-loop rollout → fixed failure-ranking model**.

The first part uses one deterministic TALOS/iCub example to show what the
method computes. The final part reads the submitted reproducibility artifact:
B5 reaches pooled failure PR-AUC 0.725 versus 0.689 for the fair B4 comparator,
but the prespecified conditional 95% interval for the macro within-robot
difference is [-0.009, 0.073]. The result is therefore a modest but
inconclusive ranking signal—not calibrated risk control or a safety guarantee.

## 1. Setup

The notebook uses only the existing feasibility API and the locked scientific environment. Plotting helpers below transform already-computed quantities for display; they do not implement dynamics or feasibility logic.


In [1]:
import json
import subprocess
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
from IPython.display import IFrame, Video, display
from meshcat.animation import Animation
from pinocchio.visualize import MeshcatVisualizer

from artifact.generate_rollout_video import generate_video_from_artifact
from src.feasibility import (
    GaitSample,
    build_whole_body_trajectory,
    compute_physics_signature,
    load_robot_spec,
    rollout,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROBOTS = ("talos", "icub")
PHASE_COLORS = {
    "left": "#4C78A8",
    "right": "#F58518",
    "touchdown": "#E45756",
    "double": "#72B7B2",
}
PHASE_LABELS = {
    "left": "L",
    "right": "R",
    "touchdown": "",
    "double": "DS",
}


def print_table(headers, rows):
    rows = [tuple(map(str, row)) for row in rows]
    widths = [
        max(len(str(header)), *(len(row[index]) for row in rows))
        for index, header in enumerate(headers)
    ]
    print(" | ".join(str(value).ljust(width) for value, width in zip(headers, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))


def draw_phase_strip(axis, times, modes):
    times = np.asarray(times)
    dt = float(np.median(np.diff(times))) if len(times) > 1 else 1.0
    start = 0
    for stop in range(1, len(modes) + 1):
        if stop == len(modes) or modes[stop] != modes[start]:
            right = times[stop] if stop < len(times) else times[-1] + dt
            axis.axvspan(
                times[start],
                right,
                color=PHASE_COLORS[modes[start]],
                linewidth=0,
            )
            width = right - times[start]
            label = PHASE_LABELS[modes[start]]
            if label:
                axis.text(
                    0.5 * (times[start] + right),
                    0.5,
                    label,
                    ha="center",
                    va="center",
                    fontsize=7,
                )
            start = stop
    axis.set_xlim(times[0], times[-1] + dt)
    axis.set_ylim(0, 1)
    axis.set_yticks([])
    axis.grid(False)


def active_min(values, active):
    selected = np.where(active, values, np.nan)
    return np.asarray([
        np.nanmin(row) if np.isfinite(row).any() else np.nan
        for row in selected
    ])


def finite_min(values):
    finite = np.asarray(values)[np.isfinite(values)]
    return float(np.min(finite)) if len(finite) else np.nan

### 1.1 Frozen validity checkpoint

The checkpoint is read from the clean frozen worktree. Only status flags and shortened fingerprints are displayed, so stored notebook outputs contain no local paths.


In [2]:
repo = Path.cwd()
common_git_text = subprocess.run(
    ["git", "rev-parse", "--git-common-dir"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
common_git = Path(common_git_text)
if not common_git.is_absolute():
    common_git = (repo / common_git).resolve()
main_repo = common_git.parent
source_repo = main_repo
evidence_dir = main_repo / "results" / "confirmation-seed42026"

try:
    validity = json.loads((evidence_dir / "validity.json").read_text())
    protocol = json.loads((evidence_dir / "protocol.json").read_text())
    head = subprocess.run(
        ["git", "-C", str(source_repo), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ["git", "-C", str(source_repo), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    test_count = sum(
        int(check["last_output_line"].split()[0])
        for check in validity["checks"]
    )
    checkpoint = {
        "available": True,
        "clean source": not dirty,
        "commit": head[:12],
        "protocol revision": protocol["pilot_revision"],
        "planned pilot / robot": protocol["pilot_rollouts"],
        "validity passed": validity["passed"],
        "validity tests": test_count,
        "environment lock": all(
            (evidence_dir / name).is_file()
            for name in (
                "environment.lock",
                "conda-explicit.txt",
                "pip-requirements.txt",
            )
        ),
        "experiment fingerprint": validity["experiment_fingerprint"][:12],
        "validity fingerprint": validity["validity_fingerprint"][:12],
    }
except (OSError, KeyError, ValueError, subprocess.SubprocessError) as error:
    checkpoint = {
        "available": False,
        "warning": type(error).__name__,
    }

print_table(
    ("checkpoint field", "value"),
    checkpoint.items(),
)


checkpoint field       | value       
-----------------------+-------------
available              | True        
clean source           | False       
commit                 | bc8d841b4586
protocol revision      | 1           
planned pilot / robot  | 200         
validity passed        | True        
validity tests         | 62          
environment lock       | True        
experiment fingerprint | 1ed3706faac6
validity fingerprint   | fa1228103c57


This checkpoint validates the physical implementation and protocol freeze. The
article-level ranking evidence is separate: it comes from two frozen,
independently scrambled confirmation sets packaged with the submission.

## 2. One robot-normalized sample

The parameters below are dimensionless with respect to each robot's capacities and morphology. In particular, step length scales with leg length \(L\), timing with \(T=\sqrt{L/g}\), width with neutral stance width, and ZMP bias with sole dimensions.

A conservative one-step example at 50 Hz keeps this walkthrough fast. It is not selected from test data and is not used for a scientific performance claim.


In [3]:
SAMPLE = GaitSample(
    step_length=0.10,
    step_width=1.0,
    single_support_duration=2.8,
    double_support_duration=0.8,
    com_height_scale=0.90,
    zmp_bias_x=0.0,
    zmp_bias_y=-0.25,
    friction=0.8,
    payload_fraction=0.0,
    timing_error_seconds=0.0,
    impulse=0.01,
    seed=2026,
)
STEPS = 10
DT = 0.01

specs = {name: load_robot_spec(name) for name in ROBOTS}
dimensional_rows = []
for name, spec in specs.items():
    gait = SAMPLE.to_gait_params(spec, steps=STEPS, dt=DT)
    dimensional_rows.append((
        name,
        f"{spec.mass:.2f}",
        f"{spec.leg_length:.3f}",
        f"{spec.natural_time:.3f}",
        f"{gait.step_length:.3f}",
        f"{gait.step_width:.3f}",
        f"{gait.single_support_duration:.3f}",
        f"{gait.double_support_duration:.3f}",
    ))

print_table(
    (
        "robot",
        "mass [kg]",
        "L [m]",
        "T [s]",
        "step [m]",
        "width [m]",
        "SS [s]",
        "DS [s]",
    ),
    dimensional_rows,
)


robot | mass [kg] | L [m] | T [s] | step [m] | width [m] | SS [s] | DS [s]
------+-----------+-------+-------+----------+-----------+--------+-------
talos | 90.27     | 0.748 | 0.276 | 0.075    | 0.170     | 0.773  | 0.221 
icub  | 28.35     | 0.474 | 0.220 | 0.047    | 0.211     | 0.615  | 0.176 


## 3. Method and evidence path

The text below separates the single-sample method walkthrough from the frozen
development and confirmation evidence. No arrows or schematic geometry encode
scientific meaning.

### Method walkthrough

1. Normalize gait and perturbation parameters by each robot's morphology and
   physical capacities.
2. Construct a manifold-aware whole-body trajectory.
3. Summarize torque, friction, CoP, impact, joint, and IK quantities.
4. Obtain the label from a separate constrained closed-loop rollout.

### Frozen evidence

5. Fit fixed B4 (parameters + robot identity) and B5 (B4 + whole-body
   signature) models on 200 development rollouts.
6. Evaluate without refitting on two independent 400-rollout confirmations.
7. Use the score only for ranking; every shortlisted gait still requires the
   full rollout oracle.

## 4. Execute the real physics path

Each robot passes through trajectory construction, the inverse-dynamics phase signature, and the independent constrained rollout exactly once. A failed rollout remains a valid diagnostic result and is not replaced by a fallback.


In [4]:
runs = {}
for name in ROBOTS:
    spec = specs[name]
    trajectory = build_whole_body_trajectory(
        spec,
        SAMPLE,
        steps=STEPS,
        dt=DT,
    )
    signature = compute_physics_signature(spec, trajectory, SAMPLE)
    result = rollout(spec, trajectory, SAMPLE)
    runs[name] = {
        "spec": spec,
        "trajectory": trajectory,
        "signature": signature,
        "rollout": result,
    }

print_table(
    ("robot", "samples", "phases", "signature features"),
    [
        (
            name,
            len(run["trajectory"].time),
            ", ".join(sorted(set(run["trajectory"].contact_modes))),
            len(run["signature"].feature_names),
        )
        for name, run in runs.items()
    ],
)


robot | samples | phases                         | signature features
------+---------+--------------------------------+-------------------
talos | 1017    | double, left, right, touchdown | 150               
icub  | 809     | double, left, right, touchdown | 150               


### 4.1 Whole-body reference trajectories

Time is normalized by \(T\), and positions by \(L\). This exposes sequence shape across morphologies while preserving each robot's dimensional trajectory in `runs`.


In [5]:
figure, axes = plt.subplots(
    3,
    2,
    figsize=(13, 7.5),
    sharey="row",
    gridspec_kw={"height_ratios": (3, 3, 0.45)},
    constrained_layout=True,
)

for column, name in enumerate(ROBOTS):
    run = runs[name]
    spec = run["spec"]
    trajectory = run["trajectory"]
    normalized_time = trajectory.time / spec.natural_time

    axes[0, column].plot(
        normalized_time,
        trajectory.com[:, 0] / spec.leg_length,
        label="CoM x/L",
        linewidth=2.0,
    )
    axes[0, column].plot(
        normalized_time,
        trajectory.left_foot[:, 0] / spec.leg_length,
        label="left foot x/L",
    )
    axes[0, column].plot(
        normalized_time,
        trajectory.right_foot[:, 0] / spec.leg_length,
        label="right foot x/L",
    )
    axes[0, column].set_title(name.upper())
    axes[0, column].set_ylabel("longitudinal / L")
    axes[0, column].legend(fontsize=8)

    axes[1, column].plot(
        normalized_time,
        trajectory.com[:, 2] / spec.leg_length,
        label="CoM z/L",
        linewidth=2.0,
    )
    axes[1, column].plot(
        normalized_time,
        trajectory.left_foot[:, 2] / spec.leg_length,
        label="left foot z/L",
    )
    axes[1, column].plot(
        normalized_time,
        trajectory.right_foot[:, 2] / spec.leg_length,
        label="right foot z/L",
    )
    axes[1, column].set_ylabel("vertical / L")
    axes[1, column].legend(fontsize=8)

    draw_phase_strip(
        axes[2, column],
        normalized_time,
        trajectory.contact_modes,
    )
    axes[2, column].set_xlabel("normalized time t/T")

figure.suptitle("Robot-normalized whole-body trajectory and contact phases")
plt.show()


/var/folders/vz/_fjds83j6z1c01cz81zyb7nr0000gn/T/ipykernel_3332/3507942287.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.2 Phase physics signature

Each signature contains five statistics for ten normalized channels in three phases (150 features). The heatmaps show the `p95` statistic for seven representative channels. The signed log transform is display-only and uses one shared color scale.


In [6]:
PHASES = ("single_support", "touchdown", "double_support")
CHANNELS = (
    "torque",
    "friction",
    "cop",
    "joint_position",
    "ik_residual",
    "dynamics_slack",
    "impact_proxy",
)


def signature_matrix(signature):
    lookup = {
        tuple(feature_name.rsplit(".", 2)): value
        for feature_name, value in zip(
            signature.feature_names,
            signature.values,
        )
    }
    return np.asarray([
        [
            lookup[(phase, channel, "p95")]
            for channel in CHANNELS
        ]
        for phase in PHASES
    ])


signature_matrices = {
    name: signature_matrix(run["signature"])
    for name, run in runs.items()
}
display_matrices = {
    name: np.sign(matrix) * np.log1p(np.abs(matrix))
    for name, matrix in signature_matrices.items()
}
shared_limit = max(
    1.0,
    max(
        np.max(np.abs(matrix))
        for matrix in display_matrices.values()
    ),
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.4),
    constrained_layout=True,
)
images = []
for axis, name in zip(axes, ROBOTS):
    image = axis.imshow(
        display_matrices[name],
        cmap="coolwarm",
        vmin=-shared_limit,
        vmax=shared_limit,
        aspect="auto",
    )
    images.append(image)
    axis.set_title(name.upper())
    axis.set_xticks(range(len(CHANNELS)), CHANNELS, rotation=35, ha="right")
    axis.set_yticks(range(len(PHASES)), PHASES)
    for row in range(len(PHASES)):
        for column in range(len(CHANNELS)):
            value = signature_matrices[name][row, column]
            axis.text(
                column,
                row,
                f"{value:.2g}",
                ha="center",
                va="center",
                fontsize=7,
                color=(
                    "white"
                    if abs(display_matrices[name][row, column])
                    > 0.55 * shared_limit
                    else "black"
                ),
            )

colorbar = figure.colorbar(
    images[-1],
    ax=axes,
    shrink=0.85,
)
colorbar.set_label("sign(x) log1p(|normalized feature|)")
figure.suptitle("Shared-scale p95 phase physics signatures")
plt.show()

signature_rows = []
for name in ROBOTS:
    signature = runs[name]["signature"]
    for feature_name, value, raw_value in list(zip(
        signature.feature_names,
        signature.values,
        signature.raw_values,
    ))[:7]:
        signature_rows.append((
            name,
            feature_name,
            f"{value:.4g}",
            f"{raw_value:.4g}",
        ))

print_table(
    ("robot", "feature", "normalized", "raw"),
    signature_rows,
)
print_table(
    ("robot", "feature shape", "solver status counts"),
    [
        (
            name,
            runs[name]["signature"].values.shape,
            dict(Counter(runs[name]["signature"].solver_status)),
        )
        for name in ROBOTS
    ],
)


robot | feature                      | normalized | raw    
------+------------------------------+------------+--------
talos | single_support.torque.min    | -0.433     | -0.9711
talos | single_support.torque.p05    | -0.4203    | -0.971 
talos | single_support.torque.median | -0.3524    | -0.9703
talos | single_support.torque.p95    | -0.3054    | -0.9696
talos | single_support.torque.max    | -0.2988    | -0.9695
talos | single_support.friction.min  | -0.9625    | -681.9 
talos | single_support.friction.p05  | -0.9524    | -674.8 
icub  | single_support.torque.min    | -0.3491    | -0.4483
icub  | single_support.torque.p05    | -0.3296    | -0.4481
icub  | single_support.torque.median | -0.185     | -0.4476
icub  | single_support.torque.p95    | -0.06534   | -0.4466
icub  | single_support.torque.max    | -0.05419   | -0.446 
icub  | single_support.friction.min  | -0.8995    | -200.1 
icub  | single_support.friction.p05  | -0.8878    | -197.5 
robot | feature shape | solver status co

/var/folders/vz/_fjds83j6z1c01cz81zyb7nr0000gn/T/ipykernel_3332/735156489.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Independent rollout checks

These traces come from the constrained closed-loop rollout—not from the reference LIPM/ZMP calculation. Torque demand is divided by actuator effort limits; friction margin by body weight; CoP margin by sole half-width. Positive contact margins are feasible, and the torque limit is 1.0.


In [7]:
def rollout_series(run):
    spec = run["spec"]
    result = run["rollout"]
    effort = np.abs(spec.effort_limits[6:])
    finite_effort = np.isfinite(effort) & (effort > 0.0)
    normalized_torque = np.divide(
        np.abs(result.torque_demand),
        effort,
        out=np.full_like(result.torque_demand, np.nan),
        where=finite_effort,
    )
    torque_ratio = np.nanmax(normalized_torque, axis=1)
    friction_ratio = (
        active_min(result.friction_margin, result.active_contacts)
        / (spec.mass * 9.81)
    )
    cop_ratio = (
        active_min(result.cop_margin, result.active_contacts)
        / spec.sole_half_width
    )
    if 0 <= result.failure_index < len(result.time):
        first_uncomputed = result.failure_index
        torque_ratio[first_uncomputed:] = np.nan
        friction_ratio[first_uncomputed:] = np.nan
        cop_ratio[first_uncomputed:] = np.nan
    return torque_ratio, friction_ratio, cop_ratio


figure, axes = plt.subplots(
    4,
    2,
    figsize=(13, 9),
    sharey="row",
    gridspec_kw={"height_ratios": (3, 3, 3, 0.45)},
    constrained_layout=True,
)
rollout_rows = []

for column, name in enumerate(ROBOTS):
    run = runs[name]
    spec = run["spec"]
    trajectory = run["trajectory"]
    result = run["rollout"]
    normalized_time = result.time / spec.natural_time
    torque_ratio, friction_ratio, cop_ratio = rollout_series(run)

    axes[0, column].plot(normalized_time, torque_ratio, color="#4C78A8")
    axes[0, column].axhline(1.0, color="#C44E52", linestyle="--", label="limit")
    axes[0, column].set_title(name.upper())
    axes[0, column].set_ylabel("max |τ| / τ_limit")
    axes[0, column].legend(fontsize=8)

    axes[1, column].plot(normalized_time, friction_ratio, color="#59A14F")
    axes[1, column].axhline(0.0, color="#C44E52", linestyle="--")
    axes[1, column].set_ylabel("friction margin / mg")

    axes[2, column].plot(normalized_time, cop_ratio, color="#B279A2")
    axes[2, column].axhline(0.0, color="#C44E52", linestyle="--")
    axes[2, column].set_ylabel("CoP margin / sole half-width")
    for row in range(3):
        axes[row, column].set_xlim(
            normalized_time[0],
            normalized_time[-1] + trajectory.dt / spec.natural_time,
        )

    if 0 <= result.failure_index < len(normalized_time):
        failure_time = normalized_time[result.failure_index]
        for row in range(3):
            axes[row, column].axvline(
                failure_time,
                color="#111827",
                linestyle=":",
                linewidth=1.5,
            )

    draw_phase_strip(
        axes[3, column],
        trajectory.time / spec.natural_time,
        trajectory.contact_modes,
    )
    axes[3, column].set_xlabel("normalized time t/T")

    rollout_rows.append((
        name,
        result.success,
        result.failure_reason or "none",
        result.failure_index,
        f"{result.runtime_seconds:.3f}",
        f"{np.nanmax(torque_ratio):.3f}",
        f"{finite_min(friction_ratio):.3e}",
        f"{finite_min(cop_ratio):.3e}",
    ))

figure.suptitle("Independent closed-loop rollout diagnostics")
plt.show()

print_table(
    (
        "robot",
        "success",
        "first failure",
        "index",
        "runtime [s]",
        "max torque ratio",
        "min friction / mg",
        "min CoP / half-width",
    ),
    rollout_rows,
)


robot | success | first failure | index | runtime [s] | max torque ratio | min friction / mg | min CoP / half-width
------+---------+---------------+-------+-------------+------------------+-------------------+---------------------
talos | True    | none          | -1    | 6.600       | 0.736            | 1.431e-02         | -1.319e-07          
icub  | True    | none          | -1    | 4.752       | 0.955            | 1.448e-05         | -1.856e-06          


/var/folders/vz/_fjds83j6z1c01cz81zyb7nr0000gn/T/ipykernel_3332/356072062.py:95: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.1 Full-mesh Meshcat replay

These live viewers replay stored closed-loop configurations; they do not call
the physics pipeline again. Keep this notebook kernel and the local Meshcat
servers running while viewing them. The static figures above remain the
portable scientific record.

The bundled Meshcat cannot merge iCub's mixed-attribute DAE primitives. Its
viewer-only model therefore sends the same loaded triangles with the DAE
`Y_UP → Z_UP` transform; the scientific robot model and rollout stay unchanged.


In [8]:
VIEWER_DT = 0.1
MESHCAT_REPETITIONS = 1000
ICUB_VISUAL_COLOR = [0.75, 0.75, 0.78, 1.0]


def replay_configurations(run):
    result = run["rollout"]
    failure_index = int(result.failure_index)
    valid_stop = (
        failure_index
        if 0 <= failure_index < len(result.q)
        else len(result.q)
    )
    valid_stop = max(1, valid_stop)
    stride = max(
        1,
        int(round(VIEWER_DT / run["trajectory"].dt)),
    )
    indices = list(range(0, valid_stop, stride))
    if indices[-1] != valid_stop - 1:
        indices.append(valid_stop - 1)
    return np.asarray(result.q)[indices].copy()


def prepare_meshcat_visuals(robot):
    dae_visuals = [
        visual
        for visual in robot.viz.visual_model.geometryObjects
        if Path(visual.meshPath).suffix.lower() == ".dae"
    ]
    if not dae_visuals:
        return False

    # Meshcat 0.x cannot merge iCub's mixed DAE attributes. Coal provides the
    # same triangles, with this Collada axis conversion applied on a viewer copy.
    y_up_to_z_up = pin.SE3(
        np.array([
            [1.0, 0.0, 0.0],
            [0.0, 0.0, -1.0],
            [0.0, 1.0, 0.0],
        ]),
        np.zeros(3),
    )
    for visual in dae_visuals:
        visual.meshPath = ""
        visual.placement = visual.placement * y_up_to_z_up
    return True


def apply_meshcat_mirrors(robot):
    for visual in robot.viz.visual_model.geometryObjects:
        scale = np.asarray(visual.meshScale).copy()
        if not np.any(scale < 0.0):
            continue
        visual.meshScale = np.abs(scale)
        mirror = np.eye(4)
        mirror[:3, :3] = np.diag(np.where(scale < 0.0, -1.0, 1.0))
        node_name = robot.viz.getViewerNodeName(
            visual,
            pin.GeometryType.VISUAL,
        )
        robot.viz.viewer[node_name]["<object>"].set_transform(mirror)


def set_looping_meshcat_animation(robot, q_sequence):
    animation = Animation(default_framerate=round(1.0 / VIEWER_DT))
    live_viewer = robot.viz.viewer
    try:
        for frame_index, q in enumerate(q_sequence):
            robot.viz.viewer = animation.at_frame(live_viewer, frame_index)
            robot.display(q)
    finally:
        robot.viz.viewer = live_viewer
    live_viewer.set_animation(
        animation,
        play=True,
        repetitions=MESHCAT_REPETITIONS,
    )


def replay_label(name, run, frame_count):
    result = run["rollout"]
    outcome = (
        "success"
        if result.success
        else f"{result.failure_reason} @ {result.failure_index}"
    )
    return (
        f"**{name.upper()}** — closed-loop rollout · "
        f"{frame_count} replay frames · {outcome}"
    )


In [9]:
name = "talos"
run = runs[name]
q_sequence = replay_configurations(run)
robot = run["spec"].robot
robot.setVisualizer(MeshcatVisualizer(), copy_models=True)
robot.initViewer(open=False)
robot.loadViewerModel(rootNodeName=f"experiment_{name}")
apply_meshcat_mirrors(robot)
robot.display(q_sequence[0])
camera = robot.viz.viewer["/Cameras/default/rotated/<object>"]
camera.set_property("zoom", 1.0 / run["spec"].leg_length)
set_looping_meshcat_animation(robot, q_sequence)
print(replay_label(name, run, len(q_sequence)))
display(IFrame(robot.viz.viewer.url(), width=1200, height=600))


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7001/static/
**TALOS** — closed-loop rollout · 103 replay frames · success


In [10]:
name = "icub"
run = runs[name]
q_sequence = replay_configurations(run)
robot = run["spec"].robot
robot.setVisualizer(MeshcatVisualizer(), copy_models=True)
uses_triangle_fallback = prepare_meshcat_visuals(robot)
robot.initViewer(open=False)
robot.loadViewerModel(
    rootNodeName=f"experiment_{name}",
    visual_color=(ICUB_VISUAL_COLOR if uses_triangle_fallback else None),
)
robot.display(q_sequence[0])
camera = robot.viz.viewer["/Cameras/default/rotated/<object>"]
camera.set_property("zoom", 1.0 / run["spec"].leg_length)
set_looping_meshcat_animation(robot, q_sequence)
print(replay_label(name, run, len(q_sequence)))
display(IFrame(robot.viz.viewer.url(), width=1200, height=600))


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/
**ICUB** — closed-loop rollout · 82 replay frames · success


### 5.2 Generate and interpret the rollout video

The video is a qualitative diagnostic, not an additional evaluation metric.
It contains four clips selected deterministically from the frozen seed-42026
confirmation set:

1. TALOS successful rollout;
2. TALOS rollout whose first failure is friction;
3. iCub successful rollout;
4. iCub rollout whose first failure is impact dynamics.

For each robot, the generator chooses a successful case with the largest step
length and a failed case that progresses furthest before its first failure.
It then rebuilds the six-step, 100 Hz reference and independently repeats the
closed-loop rollout. Generation stops if the repeated outcome disagrees with
the archived label.

- **Gray skeleton:** planned whole-body reference trajectory.
- **Green body:** actual simulated configuration in a successful rollout.
- **Red body:** actual simulated configuration in a failed rollout.
- **Simplified hulls:** visualization geometry only; they are not used by the
  feasibility oracle, surrogate, or reported metrics.

The renderer writes deterministic PNG frames with Matplotlib, encodes each
clip at 10 fps with FFmpeg/H.264, and concatenates the four clips. The complete
implementation is in
[`artifact/generate_rollout_video.py`](artifact/generate_rollout_video.py).

In [11]:
Video(str(generate_video_from_artifact("submission/ijhr/reproducibility_artifact.zip", "submission/ijhr/rollout_examples.mp4")), width=720, html_attributes="controls loop muted playsinline")

## 6. Frozen confirmation result

The submitted result is failure ranking, not thresholded risk control. The
fixed models use 200 development rollouts and are evaluated on 800 untouched
TALOS/iCub rollouts from two independent scrambled-Sobol confirmations.

In [12]:
artifact_path = repo / "submission/ijhr/reproducibility_artifact.zip"
with zipfile.ZipFile(artifact_path) as archive:
    confirmation = json.loads(archive.read("confirmation_analysis.json"))

primary = confirmation["primary"]
ranking = confirmation["heldout_ranking"]["pooled"]

confirmation_summary = {
    "B4 parameter + robot PR-AUC": f'{ranking["B4_parameters_robot"]["pr_auc"]:.3f}',
    "B5 whole-body PR-AUC": f'{ranking["B5_whole_body"]["pr_auc"]:.3f}',
    "macro within-robot difference": f'{primary["delta_pr_auc"]:+.3f}',
    "conditional 95% CI": f'[{primary["ci95"][0]:.3f}, {primary["ci95"][1]:.3f}]',
    "prespecified gate": "failed",
}
print_table(("quantity", "value"), confirmation_summary.items())

quantity                      | value          
------------------------------+----------------
B4 parameter + robot PR-AUC   | 0.689          
B5 whole-body PR-AUC          | 0.725          
macro within-robot difference | +0.034         
conditional 95% CI            | [-0.009, 0.073]
prespecified gate             | failed         


### Interpretation

Both independent scrambles have a positive point estimate, but the conditional
95% interval crosses zero. Phase resolution and rollout-budget savings are not
supported.

![Confirmation PR-AUC](submission/ijhr/figures/figure2_domain_gate.png)

![Repeated-confirmation summary](submission/ijhr/figures/figure3_heldout_ranking.png)

## 7. Takeaways

- Robot normalization makes one representation usable for TALOS and iCub,
  while the robots remain separate evaluation strata.
- The independent confirmation suggests a modest whole-body ranking signal,
  but the confidence interval crosses zero.
- Phase identity and rollout-budget savings are not supported.
- The score is a ranking aid. Every shortlisted gait still requires the full
  rollout oracle.

## 8. Reproduce the article result

The submitted JSON and figures can be regenerated from the frozen records
without running new simulations:

```sh
unzip submission/ijhr/reproducibility_artifact.zip -d reproduced
cd reproduced
python tests/test_confirmation.py
python tests/test_confirmation_analysis.py
python -m artifact.analyze_confirmation   --development-csv data/development_seed12026.csv   --development-csv data/development_seed22026.csv   --confirmation-dir data/confirmation_seed42026   --confirmation-dir data/confirmation_seed52026   --output-dir regenerated   --figures-dir regenerated/figures
```

Repeating the expensive rollouts is optional and is not required to verify the
reported analysis.